In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2'] = "true"

In [4]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-5.6-luna")

In [5]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader = WebBaseLoader('https://www.coach-tony.com/blog/10-triathlon/59-a-day-in-the-life-of-an-ironman')

In [7]:
docs = loader.load()

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents = text_splitter.split_documents(docs)

In [9]:
documents

[Document(metadata={'source': 'https://www.coach-tony.com/blog/10-triathlon/59-a-day-in-the-life-of-an-ironman', 'title': 'Coach-Tony.com - A Day in the Life of an Ironman ', 'description': 'Coach Tony, USA Triathlon (USAT) certified coach, Total Immersion Certified Swim Instructor, Slowtwitch F.I.S.T Certified Bike Fitter, Ironman certified coach', 'language': 'en-GB'}, page_content="Coach-Tony.com - A Day in the Life of an Ironman \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n \n\n\n\n\nBlog\n\n\n\n\n\n\nBike Fit\n\n\n\n\n\n\nCoaching\n\n\n\n\n\n\nAbout Coach Tony\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBlog\n\n\n\n\nFind Coach Tony's Blog Search\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n            A Day in the Life of an Ironman"),
 Document(metadata={'source': 'https://www.coach-tony.com/blog/10-triathlon/59-a-day-in-the-life-of-an-ironman', 'title': 'Coach-Tony.com - A Day in the Life of an Ironman ', 'descrip

In [10]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [11]:
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents,embeddings)

In [12]:
query = "There are many questions about Ironman and its preparation "
result = vectorstoredb.similarity_search(query)
result[0].page_content

'There are many questions about Ironman and its preparation but the most popular question seems to be "what\'s it like." What is it like to be out there 10, 12, 14+ hours and how do you survive? What do you go through and what can I expect? Why do you do it? For those of you who have completed an Ironman, you know the reward is grand; something difficult to explain in words yet radiates in your smile and demeanor from the time you cross the finish line and for the rest of your life. And so this article is a futile attempt to explain a day in the life of becoming Ironman. Futile because Ironman is a personal journey and achievement; something you need to experience to understand the glow behind the smile.'

In [17]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    Question: {input}
    """
   
)
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    Question: {input}\n    '), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x10b7af620>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10b8fc1a0>, root_client=<openai.OpenAI object at 0x10aa51010>, root_async_client=<openai.AsyncOpenAI object at 0x10b7afe00>, model_name='gpt-5.6-luna', model_kwargs={}, openai_api_key=SecretStr('**********'), op

In [14]:
retriever = vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [18]:
response = retrieval_chain.invoke({"input":"There are many questions about Ironman and its preparation "})
response['answer']

'Being an Ironman is a difficult, deeply personal journey filled with emotional highs and lows. It requires perseverance, preparation, mental toughness, determination, adaptability, and luck. Over 10–14 or more hours, most participants stop racing and focus on surviving the physical discomfort and mental doubts.\n\nDespite the pain and exhaustion, the experience is also described as an epic, enjoyable day spent doing what you love. Completing it strips away distractions and forces you to confront your deepest doubts, helping you understand yourself better. The greatest reward is crossing the finish line, hearing “You are an Ironman,” and gaining the lasting confidence that you can overcome almost anything.'